# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 24.8 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 10.7 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "QCRI/Fanar-1-9B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# Load Data

In [ ]:
import pandas as pd
data = pd.read_excel('sample_math.xlsx')

In [ ]:
data.head()

,ID,Source,Country,Group,Subject,Level,Question,Context,Answer Key,Option 1,Option 2,Option 3,Option 4,Option 5,is_few_shot
0,9294,https://www.bassmaah.com/exams/exam-attempt/2198,Jordan,STEM,Math,Primary,احد الاعداد التالية يقبل القسمة على 6 دون باق ...,NaN,A,36,56,63,NaN,NaN,0
1,9346,https://www.bassmaah.com/exams/exam-attempt/140,Jordan,STEM,Math,Primary,جد ناتج مايلي : 6 × 3981\n,NaN,C,23688,23686,23886,32886,NaN,0
2,9490,https://drive.google.com/file/d/1HcdMnL_jwJ9bv...,Palestine,STEM,Math,Primary,عدد من منزلتین مجموعهما = ١٠,NaN,A,73,71,83,NaN,NaN,0
3,9402,https://drive.google.com/file/d/11F3ZsRRecAzwy...,Palestine,STEM,Math,Primary,من أدوات القياس,NaN,C,القدم,الذراع,كلاهما,NaN,NaN,0
4,9551,https://drive.google.com/file/d/12KeOBLRQa9Dte...,Palestine,STEM,Math,Primary,أي من المسائل التالية ناتج جمعها يساوي 16,NaN,A,9+7,10+7,8+9,9+6,NaN,0


# Zero Shot

# Arabic Prompt

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة
'''
zero_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    الإجابة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
) 2),4
) 7),4
) 10),4
) 3),3
) 6),3
...,...
) 878,1
80) 70 × 80 = 5600,1
ب/ مركز\n\nالثقة: 95%,1


In [ ]:
pred_zero['question'] = data['Question']
pred_zero['Option 1'] = data['Option 1']
pred_zero['Option 2'] = data['Option 2']
pred_zero['Option 3'] = data['Option 3']
pred_zero['Option 4'] = data['Option 4']
pred_zero.to_excel('Fanar-QA-Math-ZeroShot-Arabic.xlsx', index = False)

In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_zero['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.8125    0.8000    0.8062        65
           B     0.8182    0.7500    0.7826        48
           C     0.8065    0.7463    0.7752        67
           D     0.7727    0.8500    0.8095        20
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.7750       200
   macro avg     0.6420    0.6293    0.6347       200
weighted avg     0.8079    0.7750    0.7905       200



# Pred Few Shot

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

مثال 1:
السؤال:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

الخيارات:
أ) 220 أوم
ب) 202 أوم
ج) 100 أوم
د) 484 أوم

الإجابة:
د) 484 أوم

مثال 2:
السؤال:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

الخيارات:
أ) جلوكوزيدات
ب) مستقبلات
ج) أحماض أمينية غير بروتينية
د) سيفالوسبورين

الإجابة:
ب) مستقبلات

مثال 3:
السؤال:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

الخيارات:
أ) ناتج الطرح
ب) المطروح
ج) المطروح منه
د) لاشيء مما سبق

الإجابة:
أ) ناتج الطرح
'''

few_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    الإجابة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    few_pred.append(response)

In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = few_pred
pred_few['Predicted'].value_counts()

,count
Predicted,
) 7,6
) 6,5
) 2,5
) 3,4
) 10,4
...,...
) 878,1
80\n\nالثقة: 95%,1
) قطر,1


In [ ]:
pred_few['question'] = data['Question']
pred_few['Option 1'] = data['Option 1']
pred_few['Option 2'] = data['Option 2']
pred_few['Option 3'] = data['Option 3']
pred_few['Option 4'] = data['Option 4']
pred_few.to_excel('Fanar-QA-Math-FewShot.xlsx', index = False)

In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_few['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.8154    0.8154    0.8154        65
           B     0.8750    0.7292    0.7955        48
           C     0.8644    0.7612    0.8095        67
           D     0.5312    0.8500    0.6538        20
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.7800       200
   macro avg     0.6172    0.6311    0.6148       200
weighted avg     0.8177    0.7800    0.7925       200



# CoT

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

اتبع الخطوات التالية للوصول إلى الإجابة الصحيحة:

الخطوة 1: فهم السؤال
اقرأ السؤال بعناية لفهم ما يُطلب منك تحديداً. حاول فهم الحاجة والسياق الكامن خلف السؤال.

الخطوة 2: تحليل الخيارات
اقرأ كل خيار بعناية. فكّر في معنى كل خيار وكيف يرتبط بالسؤال.

الخطوة 3: تقييم الخيارات
استخدم معرفتك ومعلوماتك ذات الصلة لتقييم كل خيار. استبعد الخيارات الخاطئة بشكل واضح، ثم قارن بين الخيارات المتبقية لتحديد الأنسب.

الخطوة 4: اختيار أفضل إجابة
اختر الخيار الأكثر منطقية وصحة بناءً على التقييم في الخطوات السابقة.

الخطوة 5: كتابة الإجابة
اكتب الإجابة الصحيحة فقط باستخدام هذا التنسيق الصارم:
أ / ب / ج / د) الإجابة

مثال 1:
السؤال:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

الخيارات:
أ) 220 أوم
ب) 202 أوم
ج) 100 أوم
د) 484 أوم

التحليل:
- الخطوة 1: فهم السؤال
المطلوب حساب المقاومة الكهربائية لمصباح قدرته 100 واط وجهده 220 فولت.
- الخطوة 2: تحليل الخيارات
لحساب المقاومة نستخدم الجهد والقدرة الكهربائية المعطاة. الخيارات هي 220 أوم، 202 أوم، 100 أوم، و484 أوم.
- الخطوة 3: تقييم الخيارات
نستخدم العلاقة التالية لحساب المقاومة:
R = V^2/P
نعوض القيم المعطاة:
R = 220^2/100 = 48400/100 = 484 أوم
- الخطوة 4: اختيار الإجابة الصحيحة
من بين الخيارات، 484 أوم هي الإجابة الصحيحة، وهي الخيار د.

الإجابة النهائية:
د) 484 أوم

مثال 2:
السؤال:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

الخيارات:
أ) جلوكوزيدات
ب) مستقبلات
ج) أحماض أمينية غير بروتينية
د) سيفالوسبورين

التحليل:
- الخطوة 1: فهم السؤال
السؤال يطلب: أي من الخيارات التالية يتشابه عمله مع عمل كرات الدم البيضاء في جسم الإنسان؟
كرات الدم البيضاء هي جزء من جهاز المناعة؛ وظيفتها الأساسية التعرف على مسببات الأمراض (بكتيريا، فيروسات، إلخ) ومهاجمتها.
- الخطوة 2: تحليل الخيارات
أ) جلوكوزيدات: مركبات سكرية؛ ليست جزءاً من جهاز المناعة.
ب) مستقبلات: المستقبلات تلعب دوراً مهماً في التعرف على مسببات الأمراض وإرسال إشارات لتحفيز الاستجابة المناعية.
ج) أحماض أمينية غير بروتينية: مركبات نادرة؛ ليست مرتبطة مباشرة بوظيفة جهاز المناعة.
د) سيفالوسبورين: مضاد حيوي يقضي على البكتيريا لكنه ليس مكوناً خلوياً في جهاز المناعة.
- الخطوة 3: تقييم الخيارات
ما الذي يشبه وظيفة كرات الدم البيضاء — أي القدرة على التعرف على مسببات الأمراض والاستجابة لها؟
المستقبلات تلعب دوراً رئيسياً في التعرف على الميكروبات وتفعيل الاستجابات المناعية.
- الخطوة 4: اختيار الإجابة الصحيحة
الخيار الأقرب لوظيفة كرات الدم البيضاء هو المستقبلات.

الإجابة النهائية:
ب) مستقبلات


مثال 3:
السؤال:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

الخيارات:
أ) ناتج الطرح
ب) المطروح
ج) المطروح منه
د) لاشيء مما سبق

التحليل:
- الخطوة 1: فهم السؤال
السؤال هو: بماذا يسمى العدد 50 في عملية الطرح 70 – 20 = 50 ؟
في عملية الطرح:
العدد الأول (70) يسمى المطروح منه.
العدد الثاني (20) يسمى المطروح.
الناتج (50) يسمى ناتج الطرح أو الفرق.
- الخطوة 2: تحليل الخيارات
ناتج الطرح: هو العدد الناتج من عملية الطرح — وهو 50 في هذه الحالة.
المطروح: هو العدد الذي يتم طرحه — أي 20، وليس 50.
المطروح منه: هو العدد الذي نطرح منه — أي 70، وليس 50.
لاشيء مما سبق: غير صحيح لأن الخيار (أ) صحيح.
- الخطوة 3: تقييم الخيارات
العدد 50 هو ناتج عملية الطرح.
- الخطوة 4: اختيار الإجابة الصحيحة
الإجابة الصحيحة هي ناتج الطرح.

الإجابة النهائية:
أ) ناتج الطرح
'''

cot_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    cot_pred.append(response)

In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = cot_pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
) 2,4
) 7,4
) 3,3
) 12,3
) 10,3
...,...
) قطر,1
") 1\n\nالتفسير:\nالعدد الأولي هو عدد طبيعي أكبر من 1 ليس له قواسم غير الواحد نفسه. العدد 1 ليس أولياً لأنه قابل للقسمة على 1 فقط، بينما جميع الأعداد الأولية الأخرى فردية. لذلك، العدد الوحيد الذي ليس أولياً وهو فردي هو 1. لكن، في الواقع، 1 ليس أولياً، وبالتالي فإن الإجابة الصحيحة هي c) 1، لكن يجب أن نلاحظ أن 1 ليس أولياً، بل هو عدد خاص. ومع ذلك، بناءً على السؤال الذي يطلب ""ما عدا""، فإن الإجابة الأكثر دقة من بين الخيارات المعطاة هي c) 1. لكن يجب أن نذكر أن 1 ليس أولياً.",1
/أ) 8,1


In [ ]:
pred_cot['question'] = data['Question']
pred_cot['Option 1'] = data['Option 1']
pred_cot['Option 2'] = data['Option 2']
pred_cot['Option 3'] = data['Option 3']
pred_cot['Option 4'] = data['Option 4']
pred_cot.to_excel('Fanar-QA-Math-CoT.xlsx', index = False)

In [ ]:
pred_cot['Predicted Normalized'].value_counts()

,count
Predicted Normalized,
A,67
C,56
B,43
D,30
Unclassified,4


In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_cot['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.7463    0.7692    0.7576        65
           B     0.8605    0.7708    0.8132        48
           C     0.8750    0.7313    0.7967        67
           D     0.5667    0.8500    0.6800        20
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.7650       200
   macro avg     0.6097    0.6243    0.6095       200
weighted avg     0.7988    0.7650    0.7763       200

